# Context lifecycle: WRITE → SELECT → COMPRESS → ISOLATE → INVALIDATE
Run the synthetic rehearsal first. It uses the real LangGraph/runtime with scripted source data and LLM responses; it does not measure model quality. Then optionally run OpenAI against live GitHub repositories. Metrics are measured per review, not illustrative token numbers.

In [ ]:
import os, subprocess, sys, importlib
from pathlib import Path

REPO_URL = "https://github.com/swarnamalya-mohan/context-engineering-long-running-agent.git"
BRANCH = "feature/initial-agent-implementation"
if not Path("src/repo_agent").exists():
    folder = Path("context-engineering-long-running-agent")
    if not folder.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(folder)], check=True)
    os.chdir(folder)

# Refresh even when a previous setup left us inside the repository.
if Path(".git").exists():
    current_branch = subprocess.check_output(
        ["git", "branch", "--show-current"], text=True).strip()
    if current_branch != BRANCH:
        raise RuntimeError(f"Expected {BRANCH}, found {current_branch}. Open a fresh Colab session.")
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
    revision = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
else:
    revision = "local source checkout"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
repo_root = Path.cwd().resolve()
src_path = str(repo_root / "src")
sys.path = [src_path] + [entry for entry in sys.path if entry != src_path]

# Discard cached package imports after updating the source.
# Re-run subsequent notebook cells to rebuild graph/runtime objects.
for module_name in list(sys.modules):
    if module_name == "repo_agent" or module_name.startswith("repo_agent."):
        del sys.modules[module_name]
importlib.invalidate_caches()
from repo_agent.offline import run_offline
from repo_agent.tools.github import candidate_relevance
assert candidate_relevance({"full_name": "test/awesome-api-clients"}) == 0
print(f"Setup complete: {repo_root}")
print(f"Code revision: {revision}; current discovery filter loaded")


## 1. Repeatable rehearsal — no key required
Run the setup cell above first and wait for `Setup complete` before running this cell.
SSO enforcement is disabled by default. Alpha and Beta pass the base spec; allowing AGPL adds Gamma. Set `ENABLE_SSO_DEMO = True` only to demonstrate the stricter SSO requirement.

In [ ]:
from repo_agent.offline import run_offline
ENABLE_SSO_DEMO = False
state = run_offline(".", enable_sso_demo=ENABLE_SSO_DEMO)
print(state["final_recommendation"])


## 2. Show actual context assembly and on-demand skills

In [ ]:
import pandas as pd
contexts = pd.DataFrame(e for e in state["events"] if e["type"] == "context_assembled")
display(contexts[["repo", "loaded_skills", "jit_files", "selected_memory_count", "raw_tokens", "filtered_tokens", "prompt_tokens"]])
contexts.plot.bar(x="repo", y=["raw_tokens", "filtered_tokens"], title="README tokens before and after filtering")


## 3. Show invalidation, reopening, and tool clearing

In [ ]:
display(pd.DataFrame(e for e in state["events"] if e["type"] in
    {"conclusion_invalidated", "openspec_change_applied", "tool_output_cleared"}))
from repo_agent.metrics import candidate_frame, evidence_frame
display(candidate_frame(state))
display(evidence_frame(state))


## 4. Clear history and recover from a durable checkpoint
The checkpoint retains policy, candidate decisions, evidence, and pending work. This demonstrates runtime state recovery; the default LangGraph checkpointer itself is in memory.

In [ ]:
import json
from repo_agent.offline import fixture_runtime
compaction = next(e for e in state["events"] if e["type"] == "compaction")
print("Transient context:", compaction["before_tokens"], "→", compaction["after_tokens"], "tokens")
recovered = json.loads(Path(compaction["durable_checkpoint"]).read_text())
recovered["working_history"] = []
assert recovered["evidence"] == state["evidence"]
print(fixture_runtime(".").synthesize(recovered))


## 5. Optional live OpenAI + GitHub run
Set `RUN_LIVE = True`. Store `OPENAI_API_KEY` and optionally `GITHUB_TOKEN` in Colab Secrets. A live run incurs API usage. Repository facts and rankings can differ from the rehearsal.

In [ ]:
RUN_LIVE = False
if RUN_LIVE:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        try:
            os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
        except Exception:
            pass
    except ImportError:
        pass  # Local execution reads environment variables.
    from repo_agent.config import Settings
    from repo_agent.graph import build_graph
    from repo_agent.reporting import decision_report
    settings = Settings.from_env()
    graph = build_graph(settings=settings, repo_root=".", enable_sso_demo=ENABLE_SSO_DEMO)
    config = {"configurable": {"thread_id": "live-conference"}}
    for update in graph.stream({}, config=config, stream_mode="updates"):
        stage, stage_state = next(iter(update.items()))
        print(stage)
        if stage in {"review_base", "review_sso", "review_agpl"}:
            print(decision_report(stage_state))
            print("\n" + "=" * 60 + "\n")
    live_state = graph.get_state(config).values
    display(candidate_frame(live_state))
    live_events = pd.DataFrame(live_state["events"])
    display(live_events)
    failures = [e for e in live_state["events"] if e["type"] == "review_error"]
    print(f"Review errors: {len(failures)}")
    if failures:
        display(pd.DataFrame(failures)[["repo", "error"]])
    print("Current candidate statuses:")
    display(candidate_frame(live_state)["status"].value_counts())
    print(live_state["final_recommendation"])


## 6. Verify behavioral invariants

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## Inspect results from a completed live run without new API calls
These reports read saved base/SSO snapshots and current final state. A base shortlist is valid only under the base requirements. Rejected candidates remain ineligible under the final policy.

In [ ]:
from repo_agent.reporting import decision_report
report_files = ["base_review_state.json"]
if ENABLE_SSO_DEMO:
    report_files.append("sso_review_state.json")
report_files.append("final_state.json")
for filename in report_files:
    snapshot = Path("artifacts") / filename
    if snapshot.exists():
        print(f"\n--- {filename} ---")
        print(decision_report(json.loads(snapshot.read_text())))
